# Attention As Kernel Operator

This notebook re-derives exact softmax attention as a row-stochastic kernel operator. The goal is to make the baseline precise before later notebooks approximate it.

## Motivation

Exact attention is the reference mechanism every sparse, low-rank, or kernelized variant must match or approximate. For queries `Q`, keys `K`, and values `V`, the attention matrix

$$A = \operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)$$

has nonnegative rows that sum to one, so `A` is a data-dependent Markov-style kernel. The output operator is `O = AV`: each query row forms a convex combination of value vectors. This is the exact baseline introduced in *Attention Is All You Need* [@attention_is_all_you_need].

## Mathematical derivation

Fix one query row `q_i` and define logits `s_{ij} = q_i^\top k_j / \sqrt{d_k}`. Let `p` be a probability vector over the keys. Consider the entropy-regularized selection problem

$$\max_{p \in \Delta^{T-1}} \; p^\top s_i + H(p), \qquad H(p) = -\sum_{j=1}^{T} p_j \log p_j.$$

The Lagrangian is

$$\mathcal{L}(p, \lambda) = \sum_j p_j s_{ij} - \sum_j p_j \log p_j + \lambda \left(1 - \sum_j p_j\right).$$

Differentiating with respect to `p_j` gives

$$s_{ij} - (1 + \log p_j) - \lambda = 0 \quad \Rightarrow \quad p_j = \exp(s_{ij} - \lambda - 1).$$

Normalizing over `j` yields

$$p_j = \frac{\exp(s_{ij})}{\sum_{\ell} \exp(s_{i\ell})} = \operatorname{softmax}(s_i)_j.$$

So softmax attention is the entropy-regularized optimum over the simplex, and the output row is `o_i = \sum_j p_j v_j`. A causal mask changes the feasible simplex by forcing future positions to carry zero mass before normalization.

In [ ]:
import math

import torch

from llm_from_scratch.functional import causal_mask
from llm_from_scratch.research import (
    attention_entropy,
    check_row_stochastic,
    exact_attention,
    finite_difference_attention_gradients,
)

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

## PyTorch implementation

The research utility implements exact attention with explicit masking and stable normalization. The returned attention matrix should stay row-stochastic so it can be treated as a normalized kernel acting on the value sequence.

In [ ]:
q = torch.tensor([[[1.0, 0.2], [0.1, 1.2], [0.9, 0.9], [0.7, -0.4]]])
k = torch.tensor([[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [1.0, -1.0]]])
v = torch.tensor([[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, -1.0]]])

output, weights = exact_attention(q, k, v)
attention_matrix = weights[0]

{
    'output_shape': tuple(output.shape),
    'weight_shape': tuple(weights.shape),
    'row_sums': attention_matrix.sum(dim=-1),
    'matrix_rank': int(torch.linalg.matrix_rank(attention_matrix).item()),
    'first_row_weights': attention_matrix[0],
}

In [ ]:
temperature_summary = []
for scale in (0.5, 3.0):
    _, scaled_weights = exact_attention(scale * q, k, v)
    entropies = attention_entropy(scaled_weights)
    temperature_summary.append(
        {
            'scale': scale,
            'first_row_weights': scaled_weights[0, 0],
            'mean_entropy': float(entropies.mean().item()),
            'max_weight': float(scaled_weights.max().item()),
        }
    )
assert temperature_summary[1]['mean_entropy'] < temperature_summary[0]['mean_entropy']
assert temperature_summary[1]['max_weight'] > temperature_summary[0]['max_weight']
temperature_summary

## Numerical checks

The baseline should satisfy structural invariants before it is used as a comparison target: rows must sum to one, masked future positions must stay zero, entropies must live in valid bounds, and autograd should agree with a central-difference check on a tiny example.

In [ ]:
mask = causal_mask(q.shape[-2]).unsqueeze(0)
_, masked_weights = exact_attention(q, k, v, mask=mask)
entropies = attention_entropy(weights)
gradient_diagnostics = finite_difference_attention_gradients(
    q.double(),
    k.double(),
    v.double(),
    eps=1e-6,
)

assert check_row_stochastic(weights)
assert check_row_stochastic(masked_weights)
assert torch.allclose(masked_weights.sum(dim=-1), torch.ones_like(masked_weights.sum(dim=-1)), atol=1e-6)
assert torch.allclose(masked_weights[0].triu(diagonal=1), torch.zeros_like(masked_weights[0].triu(diagonal=1)))
assert torch.all(entropies >= 0.0)
assert torch.all(entropies <= math.log(weights.shape[-1]) + 1e-6)
assert max(stats['max_abs_error'] for stats in gradient_diagnostics.values()) < 5e-4

{
    'entropy_per_row': entropies[0],
    'causal_future_mass': masked_weights[0].triu(diagonal=1).sum(),
    'gradient_max_abs_error': {name: round(stats['max_abs_error'], 8) for name, stats in gradient_diagnostics.items()},
}

## Exercises

Work the companion exercise sheet after running the notebook:

- `exercises/research/14_attention_as_kernel_operator.md`
- `exercises/research/solutions/14_attention_as_kernel_operator_solutions.md`

## References

- Vaswani et al., *Attention Is All You Need* [@attention_is_all_you_need].